In [0]:
"""
--------------------------------------------------------------------------------
PROJETO HACKATHON 2025 - ENGENHARIA DE DADOS
SCRIPT: 01_silver_bureau_full.py
OBJETIVO: Transformação da camada Bronze para Silver (base Spine: bureau_full).
--------------------------------------------------------------------------------
DESCRIÇÃO TÉCNICA:
Este script lê os dados da camada Bronze (Delta), aplica:
- Tipagem (casting) de colunas
- Tratamento de sentinelas (SCORE_01 = 0 -> missing)
- Padronização de flags (FLAG_INSTALACAO 0/1)
- Deduplicação pela chave de negócio (NUM_CPF + SAFRA)
e grava em Delta Lake na camada Silver.

AJUSTES UNITY CATALOG:
- Utiliza caminhos no formato /Volumes/...
- Mantém rastreabilidade via colunas metadata_*.
--------------------------------------------------------------------------------
"""

import sys
import argparse
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# Importação da função utilitária criada anteriormente em src/utils
from src.utils.spark_utils import get_spark_session


# =============================================================================
# CONFIGURAÇÃO PADRÃO (DESENVOLVIMENTO / DATABRICKS COMMUNITY)
# =============================================================================
DEFAULT_INPUT_PATH = "/Volumes/hackathon_2025/default/bronze/bureau_full_delta/"
DEFAULT_OUTPUT_PATH = "/Volumes/hackathon_2025/default/silver/bureau_full_delta/"
DEFAULT_FORMAT = "delta"
# =============================================================================


def cast_and_clean(df):
    """
    Aplica tipagem e regras mínimas da Silver para bureau_full.

    Regras:
    - SAFRA: manter como string (padrão yyyyMM)
    - NUM_CPF: string
    - FLAG_INSTALACAO: int (0/1)
    - SCORE_01: double
      - SCORE_01 == 0 => missing (SCORE_01_ADJ = NULL) e FLAG_SCORE01_MISSING=1
    - SCORE_02: (opcional) tipar para double, mas não usar na ABT v1
    - FPD/PROD/flag_mig2: manter tipado como string (ou int, se fizer sentido depois)

    Deduplicação posterior será por (NUM_CPF, SAFRA) com critério de “mais recente”
    via metadata_data_ingestao (desc).
    """

    print(">>> [Transform] Aplicando casting e regras de limpeza (Silver)...")

    df2 = (
        df
        # Normalização básica (trim)
        .withColumn("NUM_CPF", F.trim(F.col("NUM_CPF")).cast("string"))
        .withColumn("SAFRA", F.trim(F.col("SAFRA")).cast("string"))

        # FLAG_INSTALACAO: string -> int
        .withColumn("FLAG_INSTALACAO_INT", F.col("FLAG_INSTALACAO").cast("int"))

        # SCORE_01: string -> double (com tratamento de vazio)
        .withColumn("SCORE_01_DBL", F.when(F.trim(F.col("SCORE_01")) == "", None)
                    .otherwise(F.col("SCORE_01").cast("double")))

        # SCORE_02: manter tipado (opcional)
        .withColumn("SCORE_02_DBL", F.when(F.trim(F.col("SCORE_02")) == "", None)
                    .otherwise(F.col("SCORE_02").cast("double")))

        # Flags auxiliares
        .withColumn(
            "FLAG_SCORE01_MISSING",
            F.when(F.col("SCORE_01_DBL").isNull(), 1)
             .when(F.col("SCORE_01_DBL") == 0, 1)
             .otherwise(0)
        )
        .withColumn(
            "SCORE_01_ADJ",
            F.when((F.col("SCORE_01_DBL") == 0) | (F.col("SCORE_01_DBL").isNull()), None)
             .otherwise(F.col("SCORE_01_DBL"))
        )

        # Mantém colunas originais úteis (se quiser enxugar depois, é só remover)
        .withColumn("FPD", F.col("FPD").cast("string"))
        .withColumn("PROD", F.col("PROD").cast("string"))
        .withColumn("flag_mig2", F.col("flag_mig2").cast("string"))

        # Metadados (mantemos)
        .withColumn("metadata_data_ingestao", F.col("metadata_data_ingestao"))
        .withColumn("metadata_nome_arquivo_origem", F.col("metadata_nome_arquivo_origem"))
        .withColumn("metadata_sistema_origem", F.col("metadata_sistema_origem"))
    )

    # Seleção final (deixa bem explícito o contrato da Silver)
    df_out = df2.select(
        "NUM_CPF",
        "SAFRA",
        F.col("FLAG_INSTALACAO_INT").alias("FLAG_INSTALACAO"),

        # Score v1
        "SCORE_01_ADJ",
        "FLAG_SCORE01_MISSING",

        # Campos mantidos (úteis para evoluir)
        "SCORE_02_DBL",
        "FPD",
        "PROD",
        "flag_mig2",

        # Metadados
        "metadata_data_ingestao",
        "metadata_nome_arquivo_origem",
        "metadata_sistema_origem"
    )

    return df_out


def deduplicate(df):
    """
    Deduplicação por chave de negócio:
    - PARTITION BY (NUM_CPF, SAFRA)
    - ORDER BY metadata_data_ingestao DESC

    Mantém o registro mais recente por chave.
    """
    print(">>> [Quality] Deduplicando por (NUM_CPF, SAFRA) usando metadata_data_ingestao DESC...")

    w = Window.partitionBy("NUM_CPF", "SAFRA").orderBy(F.col("metadata_data_ingestao").desc())

    df_dedup = (
        df
        .withColumn("rn", F.row_number().over(w))
        .filter(F.col("rn") == 1)
        .drop("rn")
    )

    return df_dedup


def main():
    parser = argparse.ArgumentParser(description="ETL Bronze to Silver - Bureau Full")
    parser.add_argument("--input_path", help="Caminho Delta da Bronze Zone")
    parser.add_argument("--output_path", help="Caminho Delta de destino na Silver Zone")
    parser.add_argument("--format", default=DEFAULT_FORMAT, help="Formato do dataset (default: delta)")

    args_parsed, unknown_args = parser.parse_known_args()

    if args_parsed.input_path:
        args = args_parsed
    else:
        print(">>> [Config] AVISO: Rodando em modo interativo/DEV. Usando caminhos padrão.")
        class Args:
            input_path = DEFAULT_INPUT_PATH
            output_path = DEFAULT_OUTPUT_PATH
            format = DEFAULT_FORMAT
        args = Args()

    spark = get_spark_session("Silver_Bureau_Full")

    # -------------------------------------------------------------------------
    # 1) LEITURA (BRONZE)
    # -------------------------------------------------------------------------
    print(f">>> [Leitura] Lendo dados da Bronze: {args.input_path}")

    try:
        df_bronze = spark.read.format(args.format).load(args.input_path)
    except Exception as e:
        print(f"!!! ERRO CRÍTICO NA LEITURA: {e}")
        sys.exit(1)

    total_in = df_bronze.count()
    print(f">>> [Info] Total de registros na Bronze: {total_in}")

    # -------------------------------------------------------------------------
    # 2) TRANSFORMAÇÕES (SILVER)
    # -------------------------------------------------------------------------
    df_silver = cast_and_clean(df_bronze)

    # Quality checks simples (antes do dedupe)
    null_key = df_silver.filter(F.col("NUM_CPF").isNull() | (F.trim(F.col("NUM_CPF")) == "") | F.col("SAFRA").isNull() | (F.trim(F.col("SAFRA")) == "")).count()
    print(f">>> [Quality] Linhas com chave (NUM_CPF/SAFRA) nula/vazia: {null_key}")

    # -------------------------------------------------------------------------
    # 3) DEDUPE
    # -------------------------------------------------------------------------
    df_silver_dedup = deduplicate(df_silver)
    total_out = df_silver_dedup.count()
    print(f">>> [Info] Total após dedupe: {total_out} | removidas: {total_in - total_out}")

    # -------------------------------------------------------------------------
    # 4) ESCRITA (SILVER - DELTA)
    # -------------------------------------------------------------------------
    print(f">>> [Escrita] Salvando na camada Silver (Delta): {args.output_path}")

    df_silver_dedup.write \
        .format("delta") \
        .mode("overwrite") \
        .option("mergeSchema", "true") \
        .option("overwriteSchema", "true") \
        .save(args.output_path)

    # -------------------------------------------------------------------------
    # 5) AUDITORIAS FINAIS
    # -------------------------------------------------------------------------
    score_missing = df_silver_dedup.filter(F.col("FLAG_SCORE01_MISSING") == 1).count()
    flag_inst_1 = df_silver_dedup.filter(F.col("FLAG_INSTALACAO") == 1).count()

    print(">>> [Sucesso] Silver bureau_full gerada com sucesso.")
    print(f">>> [Audit] SCORE_01 missing (0 ou null): {score_missing}")
    print(f">>> [Audit] FLAG_INSTALACAO=1: {flag_inst_1}")


if __name__ == "__main__":
    main()